# JSON Loader


## Set Up the Environment

In [ ]:
%run setup.ipynb

## Document Loaders

Document loaders are used to import data from various sources into LangChain as `Document` objects. A `Document` typically includes a piece of text along with its associated metadata.

### Examples of Document Loaders:

- **Text File Loader:** Loads data from a simple `.txt` file.
- **Web Page Loader:** Retrieves the text content from any web page.
- **YouTube Video Transcript Loader:** Loads transcripts from YouTube videos.

### Functionality:

- **Load Method:** Each document loader has a `load` method that enables the loading of data as documents from a pre-configured source.
- **Lazy Load Option:** Some loaders also support a "lazy load" feature, which allows data to be loaded into memory gradually as needed.

For more detailed information, visit [LangChain's document loader documentation](https://python.langchain.com/docs/modules/data_connection/document_loaders/).


### JSON Loader

[JSON (JavaScript Object Notation)](https://en.wikipedia.org/wiki/JSON) is an open standard file format and data interchange format that uses human-readable text to store and transmit data objects consisting of attribute–value pairs and arrays (or other serializable values).

[JSON Lines](https://jsonlines.org/) is a file format where each line is a valid JSON value.

LangChain implements a [JSONLoader](https://api.python.langchain.com/en/latest/document_loaders/langchain_community.document_loaders.json_loader.JSONLoader.html) to convert JSON and JSONL data into LangChain `Document` objects. It uses a specified [`jq` schema](https://en.wikipedia.org/wiki/Jq_(programming_language)) to parse the JSON files, allowing for the extraction of specific fields into the content and metadata of the LangChain Document.

It uses the `jq` python package. Check out [this manual](https://jqlang.github.io/jq/manual/) for a detailed documentation of the `jq` syntax.

In [ ]:
import json

# Sample data dictionary similar to the one you provided but with modified contents
data = {
    'image': {'creation_timestamp': 1675549016, 'uri': 'image_of_the_meeting.jpg'},
    'is_still_participant': True,
    'joinable_mode': {'link': '', 'mode': 1},
    'magic_words': [],
    'messages': [
        {'content': 'See you soon!',
         'sender_name': 'User B',
         'timestamp_ms': 1675597571851},
        {'content': 'Thanks for the update! See you then.',
         'sender_name': 'User A',
         'timestamp_ms': 1675597435669},
        {'content': 'Actually, the green one is sold out.',
         'sender_name': 'User B',
         'timestamp_ms': 1675596277579},
        {'content': 'I was hoping to purchase the green one!',
         'sender_name': 'User A',
         'timestamp_ms': 1675595140251},
        {'content': 'I’m really interested in the green one, not the red!',
         'sender_name': 'User A',
         'timestamp_ms': 1675595109305},
        {'content': 'Here’s the $150 for it.',
         'sender_name': 'User B',
         'timestamp_ms': 1675595068468},
        {'photos': [{'creation_timestamp': 1675595059,
                     'uri': 'image_of_the_item.jpg'}],
         'sender_name': 'User B',
         'timestamp_ms': 1675595060730},
        {'content': 'It typically sells for at least $200 online',
         'sender_name': 'User B',
         'timestamp_ms': 1675595045152},
        {'content': 'How much are you asking?',
         'sender_name': 'User A',
         'timestamp_ms': 1675594799696},
        {'content': 'Good morning! $50 is far too low.',
         'sender_name': 'User B',
         'timestamp_ms': 1675577876645},
        {'content': 'Hello! I’m interested in the item you posted. I can offer $50. Let me know if that works for you. Thanks!',
         'sender_name': 'User A',
         'timestamp_ms': 1675549022673}
    ],
    'participants': [{'name': 'User A'}, {'name': 'User B'}],
    'thread_path': 'inbox/User A and User B chat',
    'title': 'User A and User B chat'
}

# Save the modified data to a JSON file
with open('../../shared_data/chat_data.json', 'w') as file:
    json.dump(data, file, indent=4)


In [ ]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(file_path="../../shared_data/chat_data.json",
                    jq_schema='.',
                    text_content=False)
docs = loader.load()

In [ ]:
len(docs)

In [ ]:
print(docs[0].page_content)
print(docs[0].metadata)

In [ ]:
print(docs[0])

In [ ]:
docs

Suppose we are interested in extracting the values under the `messages` key of the JSON data

In [ ]:
loader = JSONLoader(
    file_path='../../shared_data/chat_data.json',
    jq_schema='.messages[]',
    text_content=False)

data = loader.load()
data

Suppose we are interested in extracting the values under the `content` field within the `messages` key of the JSON data

In [ ]:
loader = JSONLoader(
    file_path='../../shared_data/chat_data.json',
    jq_schema='.messages[].content',
    text_content=False)

data = loader.load()
data

#### Basic JSON Loading
For robust loading, especially with diverse file types, consider these options:

In [ ]:
from pprint import pprint

In [ ]:
file_path = '../../shared_data/facebook_chat.json'
with open(file_path, "r") as file:
    data = json.load(file)

pprint(data)

#### Using JSONLoader for Structured Retrieval: 
Use jq_schema to specify the data structure and extract only the required fields (Schema-Based Retrieval)

In [ ]:
loader = JSONLoader(
    file_path='../../shared_data/facebook_chat.json',
    jq_schema='.messages[].content',
    text_content=False)

data = loader.load()
pprint(data)

#### Processing JSON Lines (JSONL): 
Seamlessly handle files where each line represents a separate JSON object by setting json_lines=True.

In [ ]:
# Example - JSON (Processing JSON Lines)

loader = JSONLoader(
    file_path='../../shared_data/facebook_chat_messages.jsonl',
    jq_schema=".",
    text_content=False,
    json_lines=True
)

data = loader.load()
pprint(data)

In [ ]:
# Example - JSON (Processing JSON Lines)

loader = JSONLoader(
    file_path='../../shared_data/facebook_chat_messages.jsonl',
    jq_schema='.sender_name',
    text_content=False,
    json_lines=True
)

data = loader.load()
pprint(data)

In [ ]:
# Example - JSON (Use jq_schema='.' and content_key for simpler extraction)

loader = JSONLoader(
    file_path='../../shared_data/facebook_chat_messages.jsonl',
    jq_schema='.',
    content_key="sender_name",
    text_content=False,
    json_lines=True
)

data = loader.load()
pprint(data)

#### Adding Metadata from JSON: 
Use custom functions to extract additional metadata, enhancing data context and traceability.

In [ ]:
# Example - JSON (Adding Metadata from JSON)

def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["sender_name"] = record.get("sender_name")
    metadata["timestamp_ms"] = record.get("timestamp_ms")
    return metadata

loader = JSONLoader(
    file_path='../../shared_data/facebook_chat.json',
    jq_schema='.messages[]',
    content_key="content",
    metadata_func=metadata_func # Add metadata from JSON
)

data = loader.load()
pprint(data)